In [10]:
# CS2050: Pattern Recognition and Machine Learning
# Course Project: Hand-drawn Sketch Recognition

# This notebook:
# 1. Mounts Google Drive and extracts the sketch dataset.
# 2. Trains a KNN model with HOG + LBP + edge density features.
# 3. Provides an interactive canvas to draw a sketch and predict its category.

# ## Step 1: Setup and Imports

!pip install -q scikit-learn opencv-python matplotlib tqdm ipywidgets

import os
import cv2
import numpy as np
from google.colab import drive
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from skimage.feature import hog, local_binary_pattern
from tqdm import tqdm
import matplotlib.pyplot as plt
from IPython.display import HTML, display, clear_output
import base64
from PIL import Image
import io
import ipywidgets as widgets

# ## Step 2: Mount Google Drive and Extract Dataset

drive.mount('/content/drive')

# Define paths
zip_path = "/content/drive/My Drive/Dataset_PNG.zip"  # Adjust if needed
extract_path = "/content/Dataset_PNG"

# Create extract folder
os.makedirs(extract_path, exist_ok=True)

# Unzip
import zipfile
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

dataset_path = "/content/Dataset_PNG/sketches_png/png"
print("Categories:", os.listdir(dataset_path))

# ## Step 3: Train the Model

# Feature extraction function (HOG + LBP + edge density)
def extract_features(img):
    # HOG features
    hog_feat = hog(img, orientations=9, pixels_per_cell=(16, 16),
                   cells_per_block=(2, 2), block_norm='L2-Hys')

    # LBP features
    lbp = local_binary_pattern(img, P=8, R=1, method='uniform')
    lbp_hist = np.histogram(lbp, bins=10, range=(0, 10))[0]

    # Edge density
    img_uint8 = (img * 255).astype(np.uint8)
    edges = cv2.Canny(img_uint8, 100, 200)
    edge_density = np.mean(edges) / 255

    return np.concatenate([hog_feat, lbp_hist, [edge_density]])

# Load data with features
def load_enhanced_features(dataset_path, max_samples=100):
    categories = os.listdir(dataset_path)
    X, y = [], []

    for category in tqdm(categories, desc="Loading categories"):
        cat_path = os.path.join(dataset_path, category)
        samples = 0

        for img_name in os.listdir(cat_path)[:max_samples]:
            img_path = os.path.join(cat_path, img_name)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            img = cv2.resize(img, (128, 128))
            img = img / 255.0  # Normalize
            features = extract_features(img)
            X.append(features)
            y.append(category)
            samples += 1

            if samples >= max_samples:
                break

    return np.array(X), np.array(y)

# Load data
X, y = load_enhanced_features(dataset_path, max_samples=100)
print(f"Loaded {len(X)} samples across {len(np.unique(y))} classes")

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Scale features
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train KNN
knn = KNeighborsClassifier(
    n_neighbors=15,
    weights='distance',
    metric='manhattan',
    algorithm='ball_tree',
    n_jobs=-1
)
knn.fit(X_train_scaled, y_train)

# Evaluate
y_pred = knn.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_, zero_division=0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 16.4 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Categories: ['frog', 'couch', 'hat', 'book', 'swan', 'computer monitor', 'skull', 'candle', 'camel', 'banana', 'basket', 'wheelbarrow', 'seagull', 'skyscraper', 'calculator', 'loudspeaker', 'paper clip', 'mermaid', 'mouth', 'leaf', 'boomerang', 'stapler', 'purse', 'rooster', 'eye', 'telephone', 'rabbit', 'envelope', 'train', 'parrot', 'bottle opener', 'bathtub', 'parking meter', 'walkie talkie', 'arm', 'snake', 'table', 'guitar', 'eyeglasses', 'piano', 'kangaroo', 'nose', 'grapes', 'teddy-bear', 'rainbow', 'elephant', 'saxophone', 'pipe (for smoking)', 'backpack', 'bulldozer', 'suitcase', 'pumpkin', 'wine-bottle', 't-shirt', 'panda', 'penguin', 'speed-boat', 'parachute', 'fan', 'snail', 'helicopter', 'tablelamp', 'trousers', 'knife', 'hamburger', 'beer-mug', 'armchair', 'cactus', 'sp

Loading categories:   0%|          | 0/250 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/skimage/feature/texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
Loading categories: 100%|██████████| 250/250 [03:44<00:00,  1.11it/s]


Loaded 20000 samples across 250 classes
Test Accuracy: 0.2833

Classification Report:
                    precision    recall  f1-score   support

          airplane       0.09      0.06      0.07        16
       alarm clock       0.67      0.12      0.21        16
             angel       0.00      0.00      0.00        16
               ant       0.25      0.06      0.10        16
             apple       0.64      0.44      0.52        16
               arm       0.30      0.19      0.23        16
          armchair       0.17      0.06      0.09        16
           ashtray       0.12      0.06      0.08        16
               axe       0.38      0.50      0.43        16
          backpack       0.00      0.00      0.00        16
            banana       0.25      0.31      0.28        16
              barn       0.17      0.06      0.09        16
      baseball bat       0.37      0.44      0.40        16
            basket       0.21      0.31      0.25        16
           ba

In [14]:
# ## Step 4: Interactive Sketch Drawing Demo

# **Instructions:**
# - Run this cell to open a drawing canvas.
# - Click and drag to draw black lines on a white background.
# - Click "Clear" to reset the canvas.
# - Click "Predict" to see the model’s top-5 predictions.
# - Re-run this cell to draw a new sketch.

from IPython.display import HTML, display, clear_output
import base64
from PIL import Image
import io
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

# HTML and JavaScript for the drawing canvas
canvas_html = """
<canvas id="sketchCanvas" width="448" height="448" style="border:1px solid black;"></canvas>
<br>
<button id="clearButton">Clear</button>
<button id="predictButton">Predict</button>
<script>
    var canvas = document.getElementById('sketchCanvas');
    var ctx = canvas.getContext('2d');
    ctx.fillStyle = 'white';
    ctx.fillRect(0, 0, canvas.width, canvas.height);
    ctx.strokeStyle = 'black';
    ctx.lineWidth = 2;
    var drawing = false;

    canvas.addEventListener('mousedown', function(e) {
        drawing = true;
        ctx.beginPath();
        ctx.moveTo(e.offsetX, e.offsetY);
    });

    canvas.addEventListener('mousemove', function(e) {
        if (drawing) {
            ctx.lineTo(e.offsetX, e.offsetY);
            ctx.stroke();
        }
    });

    canvas.addEventListener('mouseup', function() {
        drawing = false;
    });

    canvas.addEventListener('mouseout', function() {
        drawing = false;
    });

    document.getElementById('clearButton').addEventListener('click', function() {
        ctx.fillStyle = 'white';
        ctx.fillRect(0, 0, canvas.width, canvas.height);
    });

    document.getElementById('predictButton').addEventListener('click', function() {
        var dataURL = canvas.toDataURL('image/png');
        google.colab.kernel.invokeFunction('notebook.predict', [dataURL], {});
    });
</script>
"""

# Output widget for predictions
output_widget = widgets.Output()

# Callback function for prediction
def predict_sketch(data_url):
    with output_widget:
        clear_output()

        # Decode the base64 image
        image_data = base64.b64decode(data_url.split(',')[1])
        img = Image.open(io.BytesIO(image_data))

        # Convert to grayscale and preprocess
        img = img.convert('L')  # Grayscale, matching training
        img = img.resize((128, 128))  # Match training size
        img_np = np.array(img) / 255.0  # Normalize

        # Display the sketch
        plt.figure(figsize=(4, 4))
        plt.imshow(img_np, cmap='gray')
        plt.title("Your Drawn Sketch")
        plt.axis('off')
        plt.show()

        # Extract HOG + LBP + edge density features
        features = extract_features(img_np)
        features_scaled = scaler.transform([features])

        # Predict
        proba = knn.predict_proba(features_scaled)[0]

        # Top-5 predictions
        top5_indices = np.argsort(proba)[-5:][::-1]
        top5_labels = label_encoder.inverse_transform(top5_indices)
        top5_probs = proba[top5_indices]

        print("Top 5 Predictions:")
        for label, prob in zip(top5_labels, top5_probs):
            print(f"{label}: {prob:.4f}")

# Register the callback with Colab
from google.colab import output
output.register_callback('notebook.predict', predict_sketch)

# Display the canvas and output
display(HTML(canvas_html))
display(output_widget)

Output()

/usr/local/lib/python3.11/dist-packages/skimage/feature/texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
